In [2]:
import pandas as pd

df = pd.read_csv('grades.csv')  # Replace with your file path
print(df.head())

#this reads the file into a dataframe within pandas

  FP130 FP130X HE111 HE111S HE111W HE111X HE112 HE112S HE112V HE112W  ...  \
0     V    NaN     V    NaN    NaN    NaN   NaN    NaN     A-    NaN  ...   
1     A    NaN     A    NaN    NaN    NaN     A    NaN    NaN    NaN  ...   
2   NaN      A     V    NaN    NaN    NaN   NaN    NaN    NaN    NaN  ...   
3    C+    NaN    A-    NaN    NaN    NaN     B    NaN    NaN    NaN  ...   
4     B    NaN    A-    NaN    NaN    NaN     A    NaN    NaN    NaN  ...   

  SM221X SM223 SM239 SP211 SP211P SP211R SP212 SP212P SP212R SY110  
0    NaN   NaN   NaN   NaN      A    NaN     A    NaN    NaN     A  
1    NaN   NaN   NaN   NaN    NaN    NaN   NaN    NaN    NaN     A  
2    NaN   NaN   NaN   NaN    NaN    NaN   NaN    NaN    NaN     A  
3    NaN     C   NaN     B    NaN    NaN    B+    NaN    NaN     C  
4    NaN   NaN   NaN     B    NaN    NaN     B    NaN    NaN    B-  

[5 rows x 40 columns]


In [7]:
grade_to_numeric = {
    "A": 95, "A-": 91,
    "B+": 88, "B": 85, "B-": 81,
    "C+": 78, "C": 75, "C-": 71,
    "D+": 68, "D": 65, "D-": 61,
    "F": 55 
    # ignore validations or anything that is not a grade
}
#validations are treated at 100's, this is the used grade scale for all classes

#this function takes the last part of the grade which comes into play when there are re-takes
def convert_grade(value):
    if pd.isna(value):
        return None
    # Split on ";" and take the last part (assuming last is the actual grade)
    last_part = str(value).split(";")[-1].strip()
    return grade_to_numeric.get(last_part, None)

#apply the grade converter to all cells
df_numeric = df.map(convert_grade)

# Compute medians
medians = df_numeric.median()
#make a pretty data frame with all the calculated medians
medians_df = medians.to_frame("Median (100-pt scale)")

print(medians_df)

        Median (100-pt scale)
FP130                    88.0
FP130X                   85.0
HE111                    91.0
HE111S                   91.0
HE111W                   88.0
HE111X                   93.0
HE112                    91.0
HE112S                   95.0
HE112V                   95.0
HE112W                   88.0
HH104                    88.0
HH104X                   85.0
HH215                    91.0
HH215A                   88.0
HH215M                   88.0
HH216                    91.0
NE203                    91.0
NL110                    91.0
NN220                    88.0
NS101                    85.0
SC111                    81.0
SC112                    78.0
SC112R                   75.0
SM121A                   78.0
SM122                    81.0
SM122R                   83.0
SM122X                   85.0
SM131                    65.0
SM221                    85.0
SM221P                   91.0
SM221X                   85.0
SM223                    85.0
SM239     

In [13]:
grade_to_numeric = {
    "A": 95, "A-": 91,
    "B+": 88, "B": 85, "B-": 81,
    "C+": 78, "C": 75, "C-": 71,
    "D+": 68, "D": 65, "D-": 61,
    "F": 55,
    "V": 100
}

# Step 1: Extract final grade if multiple (e.g. "RF;B" -> "B")
def extract_final_grade(value):
    if pd.isna(value):
        return None
    return str(value).split(";")[-1].strip()

df_clean = df.applymap(extract_final_grade)

# Step 2: Use only valid grades (keys from mapping)
valid_grades = list(grade_to_numeric.keys())

# Step 3: Build summary per class
summary = []

for col in df_clean.columns:
    grades = df_clean[col].dropna()
    
    # Keep only valid grades
    grades = grades[grades.isin(valid_grades)]
    
    # Count occurrences
    grade_counts = grades.value_counts()
    
    # Average grade (numeric)
    avg_grade = grades.map(grade_to_numeric).mean()
    
    # Build row
    row = {"Class": col}
    for g in valid_grades:
        row[g] = grade_counts.get(g, 0)
    row["Total Students"] = len(grades)
    row["Average Grade"] = avg_grade
    
    summary.append(row)

# Step 4: Final DataFrame
summary_df = pd.DataFrame(summary)

# Reorder columns: Class → grades (in fixed GPA order) → Totals
summary_df = summary_df[["Class"] + valid_grades + ["Total Students", "Average Grade"]]

print(summary_df)

     Class     A    A-   B+     B   B-   C+    C   C-   D+    D  D-   F    V  \
0    FP130  1545   715  613   951  332  206  327   57   26   60   0   2  864   
1   FP130X    15    12    7    20    3    7    8    0    1    1   0   1    0   
2    HE111  1501   802  537   796  171   85  101   28    7   29   0   1  976   
3   HE111S   217    91   68    84    7    2    7    1    0    0   0   1    0   
4   HE111W    70    35   28    48   16    5   15    2    2    3   0   0    0   
5   HE111X    27    13    8     4    1    0    1    0    0    0   0   0    0   
6    HE112  1481   736  614   707  175   85  138   25   15   29   0   6    0   
7   HE112S   237    90   53    69    9    5    7    1    0    2   0   0    0   
8   HE112V   398   157   52    66   11    4    3    5    1    1   0   0    0   
9   HE112W    52    44   42    52   13   10    8    4    0    1   0   2    0   
10   HH104  1836   807  702  1151  379  217  366   91   46   62   0   8   11   
11  HH104X    19    10    4    13    9  

/tmp/ipykernel_13093/3309330547.py:16: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_clean = df.applymap(extract_final_grade)
